In [1]:
%load_ext autoreload
%autoreload 2
import numpy as np
import fmatoolbox as fma

In [8]:
#shuffleEvents(np.random.rand(1000,3),n=5)
#SS = fma.general.shuffleEvents([[0,2],[5,1],[7,3]],n=1)
#SS = fma.general.shuffleEvents([[3,0,4],[3.5,3,3],[5,4,2],[5.5,8,7],[9,0,0],[10,1,2]],n=7,intervals=[2,9.5],group=[0,0,1,0,1,1])
#SS, group = fma.general.shuffleEvents([[.5,.5,.5],[1,1,1],[2,2,2],[5,5,5],[9,9,9],[10,10,10]],n=1,intervals=[0,9.5],group=[0,0,1,0,1,1])
#SS = fma.general.shuffleEvents([[.5,.5,.5],[1,1,1],[2,2,2],[5,5,5],[9,9,9],[10,10,10]],n=7,intervals=[2,9.5])
#SS, group = fma.general.shuffleEvents([.5,1,2,5,9,10],n=1,intervals=[0,9.5],group=[0,0,1,0,1,1])
#SS, group = fma.general.shuffleEvents([.5,1,2,5,9,10],n=0,intervals=[0,9.5],group=[0,0,1,0,1,1])
SS = fma.general.shuffleEvents([[0,2],[5,1],[7,3]],n=0,intervals=[0,9.5])
SS.shape, group.shape

((3, 2, 0), (5, 0))

In [3]:
print(SS[:,:,0]); print(SS[:,:,1])

[[ 0.5  0. ]
 [ 1.   1. ]
 [ 2.5  0. ]
 [ 5.5  0. ]
 [ 6.   1. ]
 [10.   1. ]]
[[ 2.   0. ]
 [ 2.5  0. ]
 [ 4.   1. ]
 [ 5.5  0. ]
 [ 9.   1. ]
 [10.   1. ]]


In [ ]:
# test events with equal times
shuffled, group = fma.general.shuffleEvents([0,0.5,3,4,4,9],n=5,group=[0,1,0,0,1,1])

In [5]:
# test that output 'group' is correctly ordered (feature 1 = 'group')
shuffled, group = fma.general.shuffleEvents([[3,0],[3.5,0],[5,1],[5.5,0],[9,1],[10,1]],n=7,group=[0,0,1,0,1,1])
np.all(shuffled[:,1,:]==group)

np.True_

In [2]:
# test that IEI is preserved
batch_file = '/mnt/hubel-data-103/Pietro/InfraSlowNRPaper/Data/IS_intervals.batch'
session = fma.data.readBatchFile(batch_file)[0][12]
R = fma.regions.regions(session,phases='sleep.*#0',events='InfraSlowRhythm/infraslowaval')
spikes = R.spikes()

In [4]:
n = 5
iei = np.sort(np.diff(spikes[:,0],prepend=0))
shuffled = fma.general.shuffleEvents(spikes[:,0],n=n)
comp = []
d = []
for i in range(n):
    comp.append(np.mean(np.isclose(iei, np.sort(np.diff(shuffled[:,0,i],prepend=0)))))
    d.append(np.max(np.abs( iei - np.sort(np.diff(shuffled[:,0,i],prepend=0)) )))
print(np.mean(comp))
print(np.max(d))

1.0
4.547473508864641e-13


In [5]:
n = 5
groups_unique = np.unique(spikes[:,1])
iei = {g: np.sort(np.diff( spikes[spikes[:,1]==g,0], prepend=0 )) for g in groups_unique}
shuffled, group = fma.general.shuffleEvents(spikes[:,0],n=n,group=spikes[:,1])
comp = []
d = []
for i in range(n):
    for g in groups_unique:
        valid = group[:,i] == g
        comp.append(np.all(np.isclose(iei[g], np.sort(np.diff(shuffled[valid,0,i],prepend=0)))))
        d.append(np.max(np.abs( iei[g] - np.sort(np.diff(shuffled[valid,0,i],prepend=0)) )))
print(np.mean(comp))
print(np.max(d))

1.0
4.547473508864641e-13


In [3]:
# test with intervals
n = 5
spikes = R.spikes()
isa = R.eventIntervals('slownr')
spikes = fma.general.restrict(spikes,isa) # keep only spikes in ISA
groups_unique = np.unique(spikes[:,1])
spike_times, units = fma.general.shuffleEvents(spikes[:,0],group=spikes[:,1],n=n,intervals=isa)
iei = {g: np.sort(np.diff( fma.general.restrict(spikes[spikes[:,1]==g,0],isa,shift=True), prepend=0 )) for g in groups_unique}
iei_sh = {g: [np.sort(np.diff( fma.general.restrict(spike_times[units[:,l]==g,0,l],isa,shift=True), prepend=0 )) for l in range(n)] for g in groups_unique}
AA = [ np.all([ np.all(np.isclose(iei[g], iei_sh[g][l])) for g in groups_unique ])  for l in range(n)]
print(AA)
BB = [ np.mean([ np.mean(np.isclose(iei[g], iei_sh[g][l]))  for l in range(n) ])   for g in groups_unique ]

[np.True_, np.True_, np.True_, np.True_, np.True_]
